# Data Preprocessing

In this notebook we prepare the raw airline dataset for analysis and machine learning.

The preprocessing steps include:

- Loading the raw dataset
- Handling missing values
- Removing cancelled and diverted flights
- Creating the target variable (Delayed)
- Saving the cleaned dataset for further analysis

In [5]:
# Data manipulation
import pandas as pd
import numpy as np

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")
import gdown

## Load Dataset from Google Drive

The dataset is stored on Google Drive and loaded directly using a shareable download link.

In [ ]:
import gdown

file_id = "1czSA9diDpcHBECeKJ-F-aP912lqUumNN"

url = f"https://drive.google.com/uc?id={file_id}"

output_path = r"C:\Users\ASUS\Desktop\Data_Science_Internship\Final_Project\Flight_delay_prediction\Dataset\DelayedFlights1.csv"

gdown.download(url, output_path, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1czSA9diDpcHBECeKJ-F-aP912lqUumNN
From (redirected): https://drive.google.com/uc?id=1czSA9diDpcHBECeKJ-F-aP912lqUumNN&confirm=t&uuid=b6b33cf3-6708-455d-9e7b-ae49f9a5d2a0
To: C:\Users\ASUS\Desktop\Data_Science_Internship\Final_Project\Flight_delay_prediction\Dataset\DelayedFlights1.csv
 56%|█████▋    | 140M/248M [00:20<00:12, 8.87MB/s] 

In [ ]:
df = pd.read_csv("DelayedFlights.csv")

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'DelayedFlights.csv'

## Dataset Inspection
We inspect the dataset structure, data types, and missing values.

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

## Remove Unnecessary Columns

The dataset contains an extra column (`Unnamed: 0`) which is not relevant for analysis or model training.  
We remove it to clean the dataset.

In [ ]:
df = df.drop(columns=["Unnamed: 0"])
df = df.dropna(subset=["ArrDelay"])

## Missing Value Analysis

Before training machine learning models, we check for missing values in the dataset.

In [ ]:
df.isnull().sum().sort_values(ascending=False)

## Handling Missing Delay Values

Delay-related columns contain missing values because they are only recorded when a delay occurs.
For flights with no delay, these values are filled with 0.

In [ ]:
delay_cols = [
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]

df[delay_cols] = df[delay_cols].fillna(0)

## Removing Cancelled or Diverted Flights

Flights that were cancelled or diverted do not contain valid arrival delay information,
so they are removed from the dataset.

In [ ]:
df = df[df["Cancelled"] == 0]
df = df[df["Diverted"] == 0]

## Target Variable Creation

Flights delayed by more than 15 minutes are labeled as delayed.

In [ ]:
df["Delayed"] = (df["ArrDelay"] > 15).astype(int)
df["Delayed"].value_counts()

## Reduce Dataset Size

In [ ]:
df = df.sample(n=100000, random_state=42)

## Feature Engineering: Extract Departure and Arrival Hour

The scheduled departure and arrival times are stored as integers in HHMM format.
To improve model performance, we extract the **hour of departure and arrival**.

This helps the model learn patterns such as:

- Morning congestion
- Evening peak traffic
- Night flight delays

In [ ]:
df["DepHour"] = df["CRSDepTime"] // 100
df["ArrHour"] = df["CRSArrTime"] // 100

## Drop og time cols 

In [ ]:
df = df.drop(columns=["CRSDepTime", "CRSArrTime"])

## Save Processed Dataset

In [ ]:
df.to_csv(r"C:\\Users\\ASUS\\Documents\\flight_delay_project\\cleaned_flights.csv", index=False)